<div style="
    font-family: 'Poppins', sans-serif;
    background-color: #003366;
    color: white;
    width: 700px;
    padding: 50px;
    border-radius: 25px;
    margin: 40px auto;
    box-shadow: 0 8px 16px rgba(0, 0, 0, 0.4);
    text-align: center;
">
    <h1 style="font-size: 42px; margin-bottom: 25px;">Jose Carlos Fernandez</h1>
    <p style="font-size: 24px; margin: 15px 0;"><strong>Matrícula:</strong> 2024-0371</p>
    <p style="font-size: 24px; margin: 15px 0;"><strong>Fecha:</strong> 11/06/2025</p>
    <p style="font-size: 24px; margin: 15px 0;"><strong>Maestro:</strong> Persio Martinez</p>
    <p style="font-size: 24px; margin: 15px 0;"><strong>Tema:</strong> Primer Parcial</p>

</div>

# **Introduccion** 

### *Hola profe, a continuacion vera todos los obetivos relacionados a Python, como la instalacion del paquete `redis`, el dataframe con la `informacion de SQL` sobre la base de datos `BikeStore` y el envio del dataframe a `redis desde Python`... entonces nada😅, vamo al game*

---

# **Instalacion del paquete `redis`**

In [2]:
# Pude haberlo descargado desde la terminal, pero he decidido hacerlo desde aqui para que se vea todo mas claro🤑🤑

%pip install redis


   -------------------- ------------------- 1/2 [redis]
   -------------------- ------------------- 1/2 [redis]
   -------------------- ------------------- 1/2 [redis]
   -------------------- ------------------- 1/2 [redis]
   -------------------- ------------------- 1/2 [redis]
   -------------------- ------------------- 1/2 [redis]
   -------------------- ------------------- 1/2 [redis]
   -------------------- ------------------- 1/2 [redis]
   ---------------------------------------- 2/2 [redis]

Note: you may need to restart the kernel to use updated packages.


### **Todo `redi`🤣😂 entiende? porque ready... redi... redis... ok😔**

---

# **Carga de la informacion desde `SQL a Python`**

In [9]:
# Importacion de librerias
import redis
import pandas as pd
from sqlalchemy import create_engine
import zlib
import pickle

### *En el notebook de ejemplo, se usa la biblioteca `pyodbc` para la conexion... SIN EMBARGO YOOOO, voy a usar `SQL Alchemy`, ya que esta es mas flexible y potente que `pyodbc`... incluso cuando se ejecuta una conexion, la misma biblioteca sugiere que se use `SQL Alchemy`... entonces eso voy a hacer*

In [ ]:
# Cadena de conexion para SQL Server con SQLAlchemy y pyodbc
engine = create_engine(
    "mssql+pyodbc://JC-COMPUTER\\SQLEXPRESS/BikeStores?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

# Hacemos el dataframe con pandas y nuestra consulta SQL, llamando a las triples comillas para que no haya problemas en los saltos de linea
df = pd.read_sql_query("""SELECT c.category_name as Categoria, SUM(oi.quantity * oi.list_price) as TOTAL_Facturado
                            FROM [sales].[orders] AS o
                            INNER JOIN [sales].[order_items] oi ON o.order_id = oi.order_id
                            INNER JOIN [production].[products] p ON oi.product_id = p.product_id
                            INNER JOIN [production].[categories] c ON p.category_id = c.category_id
                            GROUP BY c.category_name
                            """, engine)

# Ahora vemos los primeros 5 registros del dataframe para comprobar que todo ha ido bien
df.head()

,Categoria,TOTAL_Facturado
0,Comfort Bicycles,438506.87
1,Electric Bikes,1020236.85
2,Road Bikes,1852555.60
3,Cruisers Bicycles,1109151.04
4,Mountain Bikes,2836977.00


## **🔵CONEXION ALTERNATIVA🔵**

 **Anteriormente me he conectado mediante una `CONSULTA` de tipo SQL... pero esta no es la unica forma**

**Existe una manera de conectarnos a un `SCRIPT` ya creado, sin tener que consultar o escribir lenguaje SQL en si...**

In [8]:
# Conexion a SQL Server
engine = create_engine(
    "mssql+pyodbc://JC-COMPUTER\\SQLEXPRESS/BikeStores?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

# Con esto vamos a leer el script SQL desde la ruta del archivo directamente
# Usamos una ruta absoluta para evitar problemas de directorio
with open(r'd:\OneDrive - Instituto Tecnológico de Las Américas (ITLA)\Objetivo 1 - BD 2.sql', 'r', encoding='utf-8') as file:
    sql_script = file.read()

# Ejecutamos ahora el script y cargamos el resultado en un dataframe
df2 = pd.read_sql_query(sql_script, engine)

# Hacemos un head para comprobar que se ha ejecutado correctamente
df2.head()

,Categoria,TOTAL_Facturado
0,Comfort Bicycles,438506.87
1,Electric Bikes,1020236.85
2,Road Bikes,1852555.60
3,Cruisers Bicycles,1109151.04
4,Mountain Bikes,2836977.00


### **LISTO**, me he conectado de 2 maneras diferentes, aunque obteniendo la misma informacion, esta practica me hizo indagar mas en las formas de recoleccion de informacion... ojala se ponga mejor🤑🤣

---

# **Envio de datos a la BD #5 de `redis`**

**Pasos para enviar mi data a la BD #5 de redis**

- Obviamente ya tengo mi redis corriendo
- Crear el objeto de tipo `redis`, con su host local y su puerto, para que nuestros datos se almacenen ahi
- Especificar en el parameto `bd` el numero de la Base de Datos, en este caso, el `5`
- Con `zlib` comprimimos nuestra data, ahi mismo lo serializamos a una secuencia de bytes usando el `pickle`
- Por ultimo, con `setex` le damos 60 segundos de expiracion a nuestros datos comprimidos

In [28]:
# Conexion a la base de datos #5 de Redis
r = redis.Redis(host='127.0.0.1', port=6379, db=5)

# Esta linea va a ser para serializar y comprimir el DataFrame
context = zlib.compress(pickle.dumps(df))

# Guardamos en Redis con expiracion de 60 segundos (1 minuto) | `Datos` es la clave que he elegido para almacenar el DataFrame
r.setex("Datos", 60, context)

True

**🔵EFCETIVAMENTE🔵 obtenemos el resultado `True` indicando que la operacion fue un exito**

---

# **Cargar un nuevo dataframe consultando la información desde Redis**

**Puntos clave:**

- `r` es nuestro objeto de tipo `redis`
- Por ende, usaremos la funcion `.get` para obtener nuestros `Datos` desde redis
- Cree un nuevo dataframe llamado `df_redis`, indicando que es el dataframe jalado desde redis, luego de ser comprimido y serializado

In [30]:
# Traemos el DataFrame de Redis
data = r.get("Datos")

# Deserializamos y descomprimimos el DataFrame
df_redis = pickle.loads(zlib.decompress(data))

# Vemos el head del DataFrame restaurado para comprobar que todo ha ido bien
df_redis.head()

,Categoria,TOTAL_Facturado
0,Comfort Bicycles,438506.87
1,Electric Bikes,1020236.85
2,Road Bikes,1852555.60
3,Cruisers Bicycles,1109151.04
4,Mountain Bikes,2836977.00


In [23]:
# Nuestra data comprimida se veria asi:
data

b'x\x9cmSMo\xd30\x18\xee\'e\x05mT\xe3c\x80\x84\x10\xa7\xeeR\x90z\xd9a\xd3\xc8\xba\rAY\'U\xbbN\x91\x93\xb8\x8dU\xc7\x0e\xb6\xa3Q\xc4\xa4q\xa0l\xe0\xdb\xcc\xa1\\\xb8\xec\x86\xc4\x85\x9f\xc0\x8f\xe0\xcc\x9d\x1f\xc0\r^\xa7TK\x01Gr^;\xcf\xeb<\xcf\xeb\xe7=*\xbd\x7f[\xcc\xa5C\xd7b\xc4\x02$\x1b>\x17\xb8\xd1\x13(\xc2F\xcfm"\x85\xb6\xd3\xf8\xd4,\xbf2\x87\xa6\xaeKn\xd4\x17F\xdf\xc9\xc2\tSX0De#B\x0c\xf5\xb1\x90F_\xde\xa0\xdc\x1f\xecL\xd6\x90\xae\xaf\xff\xc9p)\xf1\xe4y\x8a\xd1\x0bn\xc2b\xe2\x0f(v=\x9bc\xc1\xd7X\x12\xc5\xc3\xc9\xe9QB\x15AB\xa0!\x9c\xea\n\xecs&\x95H|e\x81\xe5\x14ht\x05\xceN!\xa7\xa6\x9d{mZy\xcf\xbc1]So\xe7\xe1\xa9\x8cL\xb8\xa0\xcb\x81\x1a\xc6V\x89.\xec\xae\x98\x93\xe3\xc9\xf7\xa2\xce\xbf4\x9dN\xe7\xc9/\x18\xe9\xd4^W\xc6;\xd9\x07\xadWZ<\xeaq\xa1\xeen\x10\x7f\xe8S\x0cd\xe7\xb7(\xf6\x95 >\xec\r\xecF\xb5\xcbQ0]\xd4Z"!\x12\xf4g3vx\xc2\x14"\xec\x1c\x14\x12\x1a\x08\xcc2\xa0\xc5\x16\x04\xdc\x17\\fR1\xf0\xd0\x17\xbd\x84\x80~\x06\xa0\xb2\xa4\xc4\xc7\xa9B+\xcb\xf2o\x17\xec\x1c^\n\xe7\xc3\x9a\xd

---

<div style="display: flex; justify-content: center; align-items: center; height: 300px;">
    <div style="background-color: #28a745; color: white; width: 600px; height: 300px; border-radius: 30px; box-shadow: 0 4px 8px rgba(0, 0, 0, 0.2); font-size: 64px; font-weight: bold; text-align: center; display: flex; justify-content: center; align-items: center;">
        FIN
        <p style="position: absolute; bottom: 15px; font-size: 14px; color: #ddd;">Jose Carlos Fernandez</p>
    </div>
</div>